# Cell Neighborhood Tokenization

- Creator: Sebastian Birk (sebastian.birk@helmholtz-munich.de).
- Date of Creation: 23.04.2024
- Date of Last Modification: 22.08.2024 (edited by Arpit Merchant)

# 1. Setup

## 1.1 Import Libraries

In [ ]:
import os
import sys
import yaml
import time
import wandb
import pickle
import warnings
import cProfile
import numpy as np
import pandas as pd
import scanpy as sc
from pathlib import Path
import scipy.sparse as sp
from typing import Dict, Union, Literal

from pyensembl import EnsemblRelease

from nichejepa import tokenizers
from nichejepa.aggregators.aggregate_neighbors import aggregate_neighbors
from nichejepa.normalizers.normalize_by_read_depth import normalize_by_read_depth
from nichejepa.preprocessors.filter_poor_quality_cells import filter_poor_quality_cells

## 1.2 Run Notebook Setup

In [ ]:
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## 1.3 Define Helper Functions for Loading AnnData

In [ ]:
def load_anndata(st_data_silver_folder_path: Union[Path, str], 
                            dataset_name: str, 
                            species: str,
                            n_batches: int) -> sc.AnnData:
    """
    Load preprocessed data stored in an AnnData object.

    Parameters
    ----------
    - st_data_silver_folder_path (Union[Path, str]): The path to the ST data silver folder.
    - dataset_name (str): The name of the dataset.
    - species (str): The species of the dataset.
    - n_batches (int): The number of batches.

    Returns:
    ----------
    - sc.AnnData: The AnnData object.

    Raises:
    ----------
    - NotImplementedError: If simulated data with multiple batches is not implemented.
    - NotImplementedError: If species other than 'sim' is not implemented.
    """
    if species == 'sim':
        # Load data
        if n_batches == 1:
            adata = sc.read_h5ad(f"{st_data_silver_folder_path}/{dataset_name}/{dataset_name}.h5ad")
        else:
            raise ValueError("Simulated data with multiple batches not implemented.")
    elif species == 'mus_musculus':
        # Load the first batch of data.
        if n_batches == 1:
            raise ValueError("Mouse data with one batch not implemented.")
        else:
            adata = sc.read_h5ad(f"{st_data_silver_folder_path}/{dataset_name}/{dataset_name}_batch1.h5ad")
    else:
        raise NotImplementedError(f"Species {species} not implemented.")

    return adata

In [ ]:
def create_base_token_dictionary(tokenizer_directory_path: str,
                            token_dictionary_file: str, 
                            adata: Dict,
                            species: str,
                            overwrite: Literal[True, False] = True) -> Dict:
    """
    Create a base token dictionary for gene IDs and save it to a file.

    Parameters:
    ----------
    - tokenizer_directory_path (str): The path to the tokenizer directory. (Used for loading geneformer token dictionary)
    - token_dictionary_file (str): The file path to save the token dictionary.
    - adata (Dict): The dictionary containing the gene IDs.
    - species (str): The species of the dataset.
    - overwrite (bool): Whether to overwrite the base token dictionary file.
    
    Returns:
    ----------
    - Dict: The token dictionary with gene IDs as keys and indices as values.
    
    Raises:
    ----------
    - NotImplementedError: If species other than 'sim' or 'mus_musculus' is not implemented.
    """
    # Load base token dictionary if it exists, else create it
    if not overwrite:
        with open(token_dictionary_file, "rb") as f:
            base_token_dict = pickle.load(f)
    else:        
        if species == 'sim':
            # Extract Ensembl IDs of genes
            gene_ids = adata.var["ensembl_id"]
            
            # Create token dictionary for simulated data
            base_token_dict = {'<pad>': 0, '<mask>': 1}
            base_token_dict.update({gene_id: idx + 2 for idx, gene_id in enumerate(gene_ids)})
        elif species == 'mus_musculus':
            # Extract Ensembl IDs of protein coding and miRNA mouse genes
            ensembl = EnsemblRelease(111, species=species)
            ensembl.download()
            ensembl.index()
            all_genes = ensembl.genes()
            protein_coding_genes = [gene for gene in all_genes if gene.biotype == "protein_coding"]
            mirna_genes = [gene for gene in all_genes if gene.biotype == "miRNA"]
            all_relevant_genes = protein_coding_genes + mirna_genes
            gene_ids = [gene.gene_id for gene in all_relevant_genes]
            
            # Create token dictionary for mouse
            base_token_dict = {'<pad>': 0, '<mask>': 1}
            base_token_dict.update({gene_id: idx + 2 for idx, gene_id in enumerate(gene_ids)})
        else:
            raise NotImplementedError(f"Species {species} not implemented.")

        with open(token_dictionary_file, "wb") as f:
            pickle.dump(base_token_dict, f)
        
    return base_token_dict

In [ ]:
def create_nichejepa_token_dictionary(nichejepa_token_dictionary_file: str,
                                      adata: sc.AnnData, 
                                      token_dict: Dict, 
                                      add_cls_tokens: Literal[True, False] = False,
                                      use_separate_cell_and_neighborhood_tokens: Literal[True, False] = False,
                                      overwrite: Literal[True, False] = False) -> Dict:
    """
    Create a NicheJEPA token dictionary.

    Parameters:
    ----------
    - nichejepa_token_dictionary_file (str): The file path to save the token dictionary.
    - adata (sc.AnnData): The AnnData object containing the dataset.
    - token_dict (Dict): The initial token dictionary.
    - add_cls_tokens (bool): Whether to add cls tokens.
    - use_separate_cell_and_neighborhood_tokens (bool): Whether to separate cell and neighborhood tokens.
    - overwrite (bool): Whether to overwrite the NicheJEPA token dictionary file.

    Returns:
    ----------
    - Dict: The updated NicheJEPA token dictionary.
    - Int: The number of gene tokens.
    """
    # Load NichJEPA token dictionary if it exists, else create it
    if not overwrite:
        with open(nichejepa_token_dictionary_file, "rb") as f:
            nichejepa_token_dict = pickle.load(f)
    else:
        # Add cls token
        if add_cls_tokens:
            def insert_at_index(d, index, key, value):
                items = list(d.items())
                items.insert(index, (key, value))
                new_d = dict(items)
                return new_d
            token_dict = insert_at_index(token_dict, 2, "<cls_cell>", 2)
            if use_separate_cell_and_neighborhood_tokens:
                token_dict = insert_at_index(token_dict, 3, "<cls_neighborhood>", 3)

        # Remove genes not in dataset
        gene_list = adata.var["ensembl_id"].tolist()
        token_dict_copy = token_dict.copy()
        for token in token_dict_copy.keys():
            if "ENS" in token and token not in gene_list:
                del(token_dict[token])
        del(token_dict_copy)

        # Use separate tokens for each gene for cell and neighborhood
        if use_separate_cell_and_neighborhood_tokens:
            nichejepa_token_dict = {
                (key + "_cell" if "ENS" in key else key): value
                for value, key in enumerate(token_dict.keys())}
            nichejepa_token_dict_copy = nichejepa_token_dict.copy()
            i = 0
            for key in nichejepa_token_dict_copy.keys():
                if "ENS" in key:
                    nichejepa_token_dict[key.replace("cell", "neighborhood")] = len(nichejepa_token_dict_copy) + i
                    i += 1
            del(nichejepa_token_dict_copy)
        else:
            nichejepa_token_dict = {
                key: value
                for value, key in enumerate(token_dict.keys())}

        # Write to disk
        with open(nichejepa_token_dictionary_file, "wb") as file:
            pickle.dump(nichejepa_token_dict, file)

    # Count number of gene tokens 
    n_gene_tokens = len([token for token in nichejepa_token_dict.keys() if "ENS" in token])

    return nichejepa_token_dict, n_gene_tokens

## 1.4 Define Helper Functions for Normalizations

In [ ]:
def normalize_gene_counts(adata: sc.AnnData, 
                          n_batches: int, 
                          norm_factor: str) -> sc.AnnData:
    """
    Normalize gene counts within cells (row-wise) in the given AnnData object.

    Parameters:
    ----------
    - adata (sc.AnnData): Annotated data matrix with gene expression information.
    - n_batches (int): Number of batches in the data.
    - norm_factor (str): List of normalization factors.

    Returns:
    ----------
    - sc.AnnData: The cell-wise gene-count normalized AnnData object.

    Raises:
    ----------
    - NotImplementedError: If `norm_factor` other than `read_depth` is not implemented.
    - NotImplementedError: If simulated data with multiple batches is not implemented.
    """
    if n_batches == 1:
        # Filter to remove poor quality cells
        print("Filtering cells.")
        adata = filter_poor_quality_cells(adata)

        # Aggregate neighborhood cell gene expression
        print("Computing spatial neighborhood graph and aggregating counts.")
        adata = aggregate_neighbors(adata,
                                    radius=27.5)

        # Normalize gene counts by norm_factor
        if norm_factor == "read_depth":
            # Normalize gene counts within each cell (row)
            adata.X = normalize_by_read_depth(adata.X, target_size=10000)
        
            # Normalize gene counts within each neighborhood (row)
            adata.layers["X_neighborhood"] = normalize_by_read_depth(adata.layers["X_neighborhood"], target_size=10000)            
        else:
            raise NotImplementedError(f"Normalization factor {norm_factor} not implemented.")
    else:
        raise NotImplementedError("Simulated data with multiple batches not implemented.")

    return adata

In [ ]:
def compute_cell_and_neighborhood_gene_means(adata: sc.AnnData) -> np.ndarray:
    """
    Compute means of gene counts across cells and neighborhoods (column-wise).

    Parameters:
    ----------
    - adata (sc.AnnData): Annotated data matrix with gene expression information.

    Returns:
    ----------
    - np.ndarray: 1D array containing the means of gene counts across cells for each gene
    - np.ndarray: 1D array containing the means of gene counts across neighborhoods for each gene
    """
    cell_gene_means = np.mean(adata.X, axis=0)
    cell_gene_means = np.array(cell_gene_means).reshape(-1)

    neighborhood_gene_means = np.mean(adata.layers["X_neighborhood"], axis=0)
    neighborhood_gene_means = np.array(neighborhood_gene_means).reshape(-1)

    return cell_gene_means, neighborhood_gene_means

In [ ]:
def compute_cell_and_neighborhood_gene_nzmeans(adata: sc.AnnData) -> np.ndarray:
    """
    Compute means of non-zero gene counts across cells and neighborhoods (column-wise).

    Parameters:
    ----------
    - adata (sc.AnnData): Annotated data matrix with gene expression information.

    Returns:
    ----------
    - np.ndarray: 1D array containing the means of non-zero gene counts across cells for each gene
    - np.ndarray: 1D array containing the means of non-zero gene counts across neighborhoods for each gene

    """
    masked_data_cells = np.ma.masked_equal(adata.X, 0)
    cell_gene_nzmeans = np.ma.mean(masked_data_cells, axis=0)
    cell_gene_nzmeans = np.array(cell_gene_nzmeans).reshape(-1)

    masked_data_neighborhood = np.ma.masked_equal(adata.layers["X_neighborhood"], 0)
    neighborhood_gene_nzmeans = np.ma.mean(masked_data_neighborhood, axis=0)
    neighborhood_gene_nzmeans = np.array(neighborhood_gene_nzmeans).reshape(-1)

    return cell_gene_nzmeans, neighborhood_gene_nzmeans

In [ ]:
def compute_cell_and_neighborhood_gene_logmeans(adata: sc.AnnData) -> np.ndarray:
    """
    Compute log-means of gene counts across cells and neighborhoods (column-wise).

    Parameters:
    ----------
    - adata (sc.AnnData): Annotated data matrix with gene expression information.

    Returns:
    ----------
    - np.ndarray: 1D array containing the log-means of gene counts across cells for each gene
    - np.ndarray: 1D array containing the log-means of gene counts across neighborhoods for each gene
    """
    sc.pp.log1p(adata)
    cell_gene_logmeans = np.mean(adata.X, axis=0)
    cell_gene_logmeans = np.array(cell_gene_logmeans).reshape(-1)

    sc.pp.log1p(adata, layer="X_neighborhood")
    neighborhood_gene_logmeans = np.mean(adata.layers["X_neighborhood"], axis=0)
    neighborhood_gene_logmeans = np.array(neighborhood_gene_logmeans).reshape(-1)

    return cell_gene_logmeans, neighborhood_gene_logmeans

# 2 Tokenize `sim1` Data

## 2.1 Configure Parameters

In [ ]:
# Get the config file for sim1 dataset
config_fname = '/lustre/scratch126/cellgen/team361/am84/NicheJEPA/reproducibility/config/cell-neighborhood-tokenization_sim1_read-depth_shifted-log.yaml'

# Read arguments from config file
with open(config_fname, "r") as f:
    config = yaml.safe_load(f)

In [ ]:
# Configure experiment arguments
tokenizer = config["experiment"]["tokenizer"]
dataset_name = config["experiment"]["dataset_name"]
species = config["experiment"]["species"]
n_batches = config["experiment"]["n_batches"]
nproc = config["experiment"]["nproc"]
overwrite = config["experiment"]["overwrite"]
print(f"Tokenizer: {tokenizer}")
print(f"Dataset: {dataset_name}")

# Configure data paths
data_directory_path = config["paths"]["data_directory_path"]
st_data_silver_folder_path = f"{data_directory_path}/silver"
st_data_gold_folder_path = f"{data_directory_path}/gold"

# Configure profiler path
profiler_path = config["paths"]["profiler_path"]

# Configure tokenization parameters
add_cls_tokens = config["tokenization"]["add_cls_tokens"]
use_separate_cell_and_neighborhood_tokens = config["tokenization"]["use_separate_cell_and_neighborhood_tokens"]
norm_method = config["tokenization"]["norm_method"]
norm_factor = config["tokenization"]["norm_factor"]
custom_attr_name_dict = config["tokenization"]["custom_attr_name_dict"]
label_cats = config["tokenization"]["label_cats"]

# Configure tokenizer paths
tokenizer_directory_path = config["paths"]["tokenizer_directory_path"]
tokenizer_folder_path = f"{tokenizer_directory_path}/{tokenizer}/{dataset_name}"
token_dictionary_file = f"{tokenizer_folder_path}/token_dictionary.pkl"
nichejepa_token_dictionary_file = f"{tokenizer_folder_path}/token_dictionary" \
                                f"{'_cls' if add_cls_tokens else '_no_cls'}" \
                                f"{'_sep' if use_separate_cell_and_neighborhood_tokens else '_no_sep'}.pkl"
os.makedirs(f"{tokenizer_folder_path}/{dataset_name}", exist_ok=True)

# Set input and output directories
input_directory = f"{st_data_silver_folder_path}/{dataset_name}"
output_directory = f"{st_data_gold_folder_path}/{tokenizer}/{dataset_name}"

## 2.2 Create token dictionaries

In [ ]:
# Load AnnData
adata = load_anndata(st_data_silver_folder_path, dataset_name, species, n_batches)

In [ ]:
# Create token dictionary
base_token_dict = create_base_token_dictionary(tokenizer_directory_path, token_dictionary_file, adata, species, overwrite)

In [ ]:
# Create nichejepa token dictionary
_, n_gene_tokens = create_nichejepa_token_dictionary(nichejepa_token_dictionary_file, adata, base_token_dict, add_cls_tokens, use_separate_cell_and_neighborhood_tokens, overwrite)

In [ ]:
# Set output file prefix    
model_input_size = n_gene_tokens * 2
output_file_prefix = f"{dataset_name}_{norm_factor}_{norm_method}_{model_input_size}"

## 2.3 Run Tokenizer with `read_depth` and `shifted_log` Normalizations

In [ ]:
# Instantiate tokenizer
tk = tokenizers.CellNeighborhoodRankTokenizer(
        model_input_size=model_input_size,
        norm_method=norm_method,
        norm_factor=norm_factor,
        nproc=nproc,
        use_separate_cell_and_neighborhood_tokens=use_separate_cell_and_neighborhood_tokens,
        token_dictionary_file=nichejepa_token_dictionary_file,
        custom_attr_name_dict=custom_attr_name_dict)

# Tokenize data 
tk.tokenize_data(input_directory=input_directory,
                    output_directory=output_directory,
                    output_file_prefix=output_file_prefix)

## 2.4 Run Tokenizer with Other Normalizations

In [ ]:
# Configure tokenized dictionary specific paths
cell_gene_means_dictionary_file = f"{tokenizer_folder_path}/cell_gene_{norm_factor}_means_dictionary.pkl"
cell_gene_nzmeans_dictionary_file = f"{tokenizer_folder_path}/cell_gene_{norm_factor}_nzmeans_dictionary.pkl"
cell_gene_logmeans_dictionary_file = f"{tokenizer_folder_path}/cell_gene_{norm_factor}_logmeans_dictionary.pkl"
neighborhood_gene_means_dictionary_file = f"{tokenizer_folder_path}/neighborhood_gene_{norm_factor}_means_dictionary.pkl"
neighborhood_gene_nzmeans_dictionary_file = f"{tokenizer_folder_path}/neighborhood_gene_{norm_factor}_nzmeans_dictionary.pkl"
neighborhood_gene_logmeans_dictionary_file = f"{tokenizer_folder_path}/neighborhood_gene_{norm_factor}_logmeans_dictionary.pkl"

In [ ]:
# Normalize gene counts
adata = normalize_gene_counts(adata, n_batches, norm_factor)

In [ ]:
# Compute means, non-zero means, and log-means
cell_gene_means, neighborhood_gene_means = compute_cell_and_neighborhood_gene_means(adata)
cell_gene_nzmeans, neighborhood_gene_nzmeans = compute_cell_and_neighborhood_gene_nzmeans(adata)
cell_gene_logmeans, neighborhood_gene_logmeans = compute_cell_and_neighborhood_gene_logmeans(adata)

In [ ]:
# Create norm factor dictionaries for tokenizer and store on disk
cell_gene_means_dict = {}
cell_gene_nzmeans_dict = {}
cell_gene_logmeans_dict = {}
neighborhood_gene_means_dict = {}
neighborhood_gene_nzmeans_dict = {}
neighborhood_gene_logmeans_dict = {}

for (gene,
     cell_gene_mean,
     cell_gene_nzmedian,
     cell_gene_logmean,
     neighborhood_gene_mean,
     neighborhood_gene_nzmedian,
     neighborhood_gene_logmean) in zip(
        adata.var["ensembl_id"].values,
        cell_gene_means,
        cell_gene_nzmeans,
        cell_gene_logmeans,
        neighborhood_gene_means,
        neighborhood_gene_nzmeans,
        neighborhood_gene_logmeans):
    cell_gene_means_dict[gene] = cell_gene_mean
    cell_gene_nzmeans_dict[gene] = cell_gene_nzmedian
    cell_gene_logmeans_dict[gene] = cell_gene_logmean
    neighborhood_gene_means_dict[gene] = neighborhood_gene_mean
    neighborhood_gene_nzmeans_dict[gene] = neighborhood_gene_nzmedian
    neighborhood_gene_logmeans_dict[gene] = neighborhood_gene_logmean
    
# Save norm_factor dictionaries to disk
with open(cell_gene_means_dictionary_file, "wb") as file:
    pickle.dump(cell_gene_means_dict, file)
with open(cell_gene_nzmeans_dictionary_file, "wb") as file:
    pickle.dump(cell_gene_nzmeans_dict, file)
with open(cell_gene_logmeans_dictionary_file, "wb") as file:
    pickle.dump(cell_gene_logmeans_dict, file)
with open(neighborhood_gene_means_dictionary_file, "wb") as file:
    pickle.dump(neighborhood_gene_means_dict, file)
with open(neighborhood_gene_nzmeans_dictionary_file, "wb") as file:
    pickle.dump(neighborhood_gene_nzmeans_dict, file)
with open(neighborhood_gene_logmeans_dictionary_file, "wb") as file:
    pickle.dump(neighborhood_gene_logmeans_dict, file)    

In [ ]:
# Instantiate tokenizer
tk = tokenizers.CellNeighborhoodRankTokenizer(
        model_input_size=model_input_size,
        norm_method=norm_method,
        norm_factor=norm_factor,
        nproc=nproc,
        use_separate_cell_and_neighborhood_tokens=use_separate_cell_and_neighborhood_tokens,
        cell_gene_means_file=cell_gene_means_dictionary_file,
        cell_gene_nzmeans_file=cell_gene_nzmeans_dictionary_file,
        cell_gene_logmeans_file=cell_gene_logmeans_dictionary_file,
        neighborhood_gene_means_file=neighborhood_gene_means_dictionary_file,
        neighborhood_gene_nzmeans_file=neighborhood_gene_nzmeans_dictionary_file,
        neighborhood_gene_logmeans_file=neighborhood_gene_logmeans_dictionary_file,
        token_dictionary_file=nichejepa_token_dictionary_file,
        custom_attr_name_dict=custom_attr_name_dict)

# Tokenize data 
tk.tokenize_data(input_directory=input_directory,
                    output_directory=output_directory,
                    output_file_prefix=output_file_prefix)

# 3. Tokenize `merfish_mouse_brain` Data

## 3.1 Configure Parameters

In [ ]:
# Get the config file for merfish small dataset
config_fname = '/lustre/scratch126/cellgen/team361/am84/NicheJEPA/reproducibility/config/cell-neighborhood-tokenization_merfish_mouse_brain_read-depth_shifted-log.yaml'

# Read arguments from config file
with open(config_fname, "r") as f:
    config = yaml.safe_load(f)

In [ ]:
# Initialize wandb run
wandb.init(project=config["wandb"]["project"],
                    job_type=config["wandb"]["job_type"],
                    tags=config["wandb"]["tags"],
                    mode=config["wandb"]["mode"],
                    config=config)
    
# Configure experiment arguments
tokenizer = config["experiment"]["tokenizer"]
dataset_name = config["experiment"]["dataset_name"]
species = config["experiment"]["species"]
n_batches = config["experiment"]["n_batches"]
nproc = config["experiment"]["nproc"]
overwrite = config["experiment"]["overwrite"]
print(f"Tokenizer: {tokenizer}")
print(f"Dataset: {dataset_name}")

# Configure data paths
data_directory_path = config["paths"]["data_directory_path"]
st_data_silver_folder_path = f"{data_directory_path}/silver"
st_data_gold_folder_path = f"{data_directory_path}/gold"

# Configure profiler path
profiler_path = config["paths"]["profiler_path"]

# Configure tokenization parameters
add_cls_tokens = config["tokenization"]["add_cls_tokens"]
use_separate_cell_and_neighborhood_tokens = config["tokenization"]["use_separate_cell_and_neighborhood_tokens"]
norm_method = config["tokenization"]["norm_method"]
norm_factor = config["tokenization"]["norm_factor"]
custom_attr_name_dict = config["tokenization"]["custom_attr_name_dict"]
label_cats = config["tokenization"]["label_cats"]

# Configure tokenizer paths
tokenizer_directory_path = config["paths"]["tokenizer_directory_path"]
tokenizer_folder_path = f"{tokenizer_directory_path}/{tokenizer}/{dataset_name}"
token_dictionary_file = f"{tokenizer_folder_path}/token_dictionary.pkl"
nichejepa_token_dictionary_file = f"{tokenizer_folder_path}/token_dictionary" \
                                f"{'_cls' if add_cls_tokens else '_no_cls'}" \
                                f"{'_sep' if use_separate_cell_and_neighborhood_tokens else '_no_sep'}.pkl"
os.makedirs(f"{tokenizer_folder_path}/{dataset_name}", exist_ok=True)

# Set input and output directories
input_directory = f"{st_data_silver_folder_path}/{dataset_name}"
output_directory = f"{st_data_gold_folder_path}/{tokenizer}/{dataset_name}"

## 3.2 Create token dictionaries

In [ ]:
# Load AnnData
adata = load_anndata(st_data_silver_folder_path, dataset_name, species, n_batches)

In [ ]:
# Create token dictionary
base_token_dict = create_base_token_dictionary(tokenizer_directory_path, token_dictionary_file, adata, species, overwrite)

In [ ]:
# Create nichejepa token dictionary
_, n_gene_tokens = create_nichejepa_token_dictionary(nichejepa_token_dictionary_file, adata, base_token_dict, add_cls_tokens, use_separate_cell_and_neighborhood_tokens, overwrite)

In [ ]:
# Set output file prefix    
model_input_size = n_gene_tokens * 2
output_file_prefix = f"{dataset_name}_{norm_factor}_{norm_method}_{model_input_size}"

## 3.3 Run Tokenizer with `read_depth` and `shifted_log` Normalizations

In [ ]:
# Instantiate tokenizer
tk = tokenizers.CellNeighborhoodRankTokenizer(
        model_input_size=model_input_size,
        norm_method=norm_method,
        norm_factor=norm_factor,
        nproc=nproc,
        use_separate_cell_and_neighborhood_tokens=use_separate_cell_and_neighborhood_tokens,
        token_dictionary_file=nichejepa_token_dictionary_file,
        custom_attr_name_dict=custom_attr_name_dict)

# Tokenize data 
tk.tokenize_data(input_directory=input_directory,
                    output_directory=output_directory,
                    output_file_prefix=output_file_prefix)

## 3.4 Run Tokenizer with Other Normalizations

In [ ]:
# Configure tokenized dictionary specific paths
cell_gene_means_dictionary_file = f"{tokenizer_folder_path}/cell_gene_{norm_factor}_means_dictionary.pkl"
cell_gene_nzmeans_dictionary_file = f"{tokenizer_folder_path}/cell_gene_{norm_factor}_nzmeans_dictionary.pkl"
cell_gene_logmeans_dictionary_file = f"{tokenizer_folder_path}/cell_gene_{norm_factor}_logmeans_dictionary.pkl"
neighborhood_gene_means_dictionary_file = f"{tokenizer_folder_path}/neighborhood_gene_{norm_factor}_means_dictionary.pkl"
neighborhood_gene_nzmeans_dictionary_file = f"{tokenizer_folder_path}/neighborhood_gene_{norm_factor}_nzmeans_dictionary.pkl"
neighborhood_gene_logmeans_dictionary_file = f"{tokenizer_folder_path}/neighborhood_gene_{norm_factor}_logmeans_dictionary.pkl"

In [ ]:
# Normalize gene counts
adata = normalize_gene_counts(adata, n_batches, norm_factor)

In [ ]:
# Compute means, non-zero means, and log-means
cell_gene_means, neighborhood_gene_means = compute_cell_and_neighborhood_gene_means(adata)
cell_gene_nzmeans, neighborhood_gene_nzmeans = compute_cell_and_neighborhood_gene_nzmeans(adata)
cell_gene_logmeans, neighborhood_gene_logmeans = compute_cell_and_neighborhood_gene_logmeans(adata)

In [ ]:
# Create norm factor dictionaries for tokenizer and store on disk
cell_gene_means_dict = {}
cell_gene_nzmeans_dict = {}
cell_gene_logmeans_dict = {}
neighborhood_gene_means_dict = {}
neighborhood_gene_nzmeans_dict = {}
neighborhood_gene_logmeans_dict = {}

for (gene,
     cell_gene_mean,
     cell_gene_nzmedian,
     cell_gene_logmean,
     neighborhood_gene_mean,
     neighborhood_gene_nzmedian,
     neighborhood_gene_logmean) in zip(
        adata.var["ensembl_id"].values,
        cell_gene_means,
        cell_gene_nzmeans,
        cell_gene_logmeans,
        neighborhood_gene_means,
        neighborhood_gene_nzmeans,
        neighborhood_gene_logmeans):
    cell_gene_means_dict[gene] = cell_gene_mean
    cell_gene_nzmeans_dict[gene] = cell_gene_nzmedian
    cell_gene_logmeans_dict[gene] = cell_gene_logmean
    neighborhood_gene_means_dict[gene] = neighborhood_gene_mean
    neighborhood_gene_nzmeans_dict[gene] = neighborhood_gene_nzmedian
    neighborhood_gene_logmeans_dict[gene] = neighborhood_gene_logmean
    
# Save norm_factor dictionaries to disk
with open(cell_gene_means_dictionary_file, "wb") as file:
    pickle.dump(cell_gene_means_dict, file)
with open(cell_gene_nzmeans_dictionary_file, "wb") as file:
    pickle.dump(cell_gene_nzmeans_dict, file)
with open(cell_gene_logmeans_dictionary_file, "wb") as file:
    pickle.dump(cell_gene_logmeans_dict, file)
with open(neighborhood_gene_means_dictionary_file, "wb") as file:
    pickle.dump(neighborhood_gene_means_dict, file)
with open(neighborhood_gene_nzmeans_dictionary_file, "wb") as file:
    pickle.dump(neighborhood_gene_nzmeans_dict, file)
with open(neighborhood_gene_logmeans_dictionary_file, "wb") as file:
    pickle.dump(neighborhood_gene_logmeans_dict, file)    

In [ ]:
# Instantiate tokenizer
tk = tokenizers.CellNeighborhoodRankTokenizer(
        model_input_size=model_input_size,
        norm_method=norm_method,
        norm_factor=norm_factor,
        nproc=nproc,
        use_separate_cell_and_neighborhood_tokens=use_separate_cell_and_neighborhood_tokens,
        cell_gene_means_file=cell_gene_means_dictionary_file,
        cell_gene_nzmeans_file=cell_gene_nzmeans_dictionary_file,
        cell_gene_logmeans_file=cell_gene_logmeans_dictionary_file,
        neighborhood_gene_means_file=neighborhood_gene_means_dictionary_file,
        neighborhood_gene_nzmeans_file=neighborhood_gene_nzmeans_dictionary_file,
        neighborhood_gene_logmeans_file=neighborhood_gene_logmeans_dictionary_file,
        token_dictionary_file=nichejepa_token_dictionary_file,
        custom_attr_name_dict=custom_attr_name_dict)

# Tokenize data 
tk.tokenize_data(input_directory=input_directory,
                    output_directory=output_directory,
                    output_file_prefix=output_file_prefix)

# 1. Previous Setup

### 1.1 Import Libraries

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
import os
import pickle
import random
import warnings
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scgpt as scg
import scipy.sparse as sp
import seaborn as sns
import squidpy as sq
import torch
from datasets import load_from_disk
from pyensembl import EnsemblRelease
from scipy.stats import spearmanr
from scgpt.model import TransformerModel
from scgpt.tokenizer.gene_tokenizer import GeneVocab
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
from skmisc.loess import loess

from nichejepa import tokenizers
from nichejepa import utils
from nichejepa.models import get_1d_sincos_pos_embed_from_grid

### 1.2 Run Notebook Setup

In [3]:
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

### 1.3 Configure Paths and Create Directories

In [4]:
# Define paths
st_data_folder_path = "../../datasets/st_data" # spatially resolved transcriptomics data
st_data_bronze_folder_path = f"{st_data_folder_path}/bronze"
st_data_silver_folder_path = f"{st_data_folder_path}/silver"
st_data_gold_folder_path = f"{st_data_folder_path}/gold"
tokenizer = "cell_neighborhood_tokenizer"
tokenizer_folder_path = f"../../datasets/tokenizer/{tokenizer}"
results_folder_path = "../../datasets/results"
geneformer_token_dictionary_file = f"../../datasets/tokenizer/geneformer_token_dictionary.pkl"
scgpt_model_dir = Path("../../artifacts/scgpt/pretrained_model")
scgpt_model_config_file = scgpt_model_dir / "args.json"
scgpt_model_file = scgpt_model_dir / "best_model.pt"
scgpt_vocab_file = scgpt_model_dir / "vocab.json"
scgpt_metadata_file = Path("../../artifacts/scgpt/metadata/gene_info.csv")

## 2. Tokenization

In [6]:
# Define dataset
#dataset_name = "nanostring_cosmx_human_brain"
#dataset_name = "sim1_1105fts_10000locs"
dataset_name = "merfish_mouse_brain"
#dataset_name = "merfish_mouse_hypothalamus"
#dataset_name = "xenium_human_brain_cancer"

In [7]:
add_cls_tokens = False
separate_cell_and_neighborhood_tokens = False

if dataset_name == "nanostring_cosmx_human_brain":
    n_batches = 1
    species = "homo_sapiens"
    norm_factors = ["read_depth", "cell_area", "unnormalized"]
    custom_attr_name_dict = {"cell_type": "cell_types",
                             "niche_type": "niche_types"}
    label_cats = ["cell_types", "niche_types"]
elif dataset_name == "merfish_mouse_hypothalamus":
    n_batches = 5
    species = "mus_musculus"
    norm_factors = ["prenormalized"] # already normalized by cell volume
    custom_attr_name_dict = {"cell_type": "cell_types",
                             "niche_type": "niche_types"}
    label_cats = ["cell_types", "niche_types"]
elif dataset_name == "xenium_human_brain_cancer":
    n_batches = 1
    species = "homo_sapiens"
    norm_factors = ["read_depth", "cell_area"]
    custom_attr_name_dict = {"cluster": "cluster"}
    label_cats = ["cluster"]
elif "sim" in dataset_name:
    n_batches = 1
    species = "sim"
    norm_factors = ["read_depth", "unnormalized"]
    custom_attr_name_dict = {"cell_types": "cell_types",
                             "niche_types": "niche_types"}
    label_cats = ["cell_types", "niche_types"]
elif dataset_name == "merfish_mouse_brain":
    n_batches = 239
    species = "mus_musculus"
    norm_factors = ["read_depth", "unnormalized"]
    custom_attr_name_dict = {"cell_type": "cell_types",
                             "tissue": "niche_types",
                             "donor_id": "donor_id",
                             "brain_section_label": "sample_id",
                             "batch": "batch_id"}
    label_cats = ["cell_types", "niche_types"]

os.makedirs(f"{tokenizer_folder_path}/{dataset_name}", exist_ok=True)

# Set dataset-specific paths and parameters
cell_gene_means_dictionary_file = f"{tokenizer_folder_path}/{dataset_name}/cell_gene_means_dictionary.pkl"
cell_gene_nzmedians_dictionary_file = f"{tokenizer_folder_path}/{dataset_name}/cell_gene_nzmedians_dictionary.pkl"
cell_gene_logmeans_dictionary_file = f"{tokenizer_folder_path}/{dataset_name}/cell_gene_logmeans_dictionary.pkl"
neighborhood_gene_means_dictionary_file = f"{tokenizer_folder_path}/{dataset_name}/neighborhood_gene_means_dictionary.pkl"
neighborhood_gene_nzmedians_dictionary_file = f"{tokenizer_folder_path}/{dataset_name}/neighborhood_gene_nzmedians_dictionary.pkl"
neighborhood_gene_logmeans_dictionary_file = f"{tokenizer_folder_path}/{dataset_name}/neighborhood_gene_logmeans_dictionary.pkl"
nichejepa_token_dictionary_file = f"{tokenizer_folder_path}/{dataset_name}/token_dictionary" \
                                  f"{'_cls' if add_cls_tokens else '_no_cls'}" \
                                  f"{'_sep' if separate_cell_and_neighborhood_tokens else '_no_sep'}.pkl"
token_dictionary_file = f"{tokenizer_folder_path}/token_dictionary_{species}.pkl"
input_directory = f"{st_data_silver_folder_path}/{dataset_name}"
output_directory = f"{st_data_gold_folder_path}/{tokenizer}/{dataset_name}"

### 2.1 Create Token Dictionary

In [8]:
if species == "mus_musculus":
    # Extract Ensembl IDs of protein coding and miRNA mouse genes
    ensembl = EnsemblRelease(111, species=species)
    ensembl.download()
    ensembl.index()
    all_genes = ensembl.genes()
    protein_coding_genes = [gene for gene in all_genes if gene.biotype == "protein_coding"]
    mirna_genes = [gene for gene in all_genes if gene.biotype == "miRNA"]
    all_relevant_genes = protein_coding_genes + mirna_genes
    gene_ids = [gene.gene_id for gene in all_relevant_genes]
    
    # Create token dictionary for mouse
    token_dict = {'<pad>': 0, '<mask>': 1}
    token_dict.update({gene_id: idx + 2 for idx, gene_id in enumerate(gene_ids)})
    with open(token_dictionary_file, "wb") as f:
        pickle.dump(token_dict, f)
elif species == "homo_sapiens":
    # Load geneformer token dict to get protein-coding and miRNA genes for human
    with open(geneformer_token_dictionary_file, "rb") as f:
        token_dict = pickle.load(f)
    with open(token_dictionary_file, "wb") as f:
            pickle.dump(token_dict, f)
elif species == "sim":
    # Load data
    if n_batches == 1:
        adata = sc.read_h5ad(f"{st_data_silver_folder_path}/{dataset_name}/{dataset_name}.h5ad")
    else:
        adata = sc.read_h5ad(f"{st_data_silver_folder_path}/{dataset_name}/{dataset_name}_batch1.h5ad")

    gene_ids = adata.var["ensembl_id"]
    token_dict = {'<pad>': 0, '<mask>': 1}
    token_dict.update({gene_id: idx + 2 for idx, gene_id in enumerate(gene_ids)})
    with open(token_dictionary_file, "wb") as f:
        pickle.dump(token_dict, f)

INFO:pyensembl.download_cache:Fetching /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCm39/ensembl111/Mus_musculus.GRCm39.111.gtf.gz from URL https://ftp.ensembl.org/pub/release-111/gtf/mus_musculus/Mus_musculus.GRCm39.111.gtf.gz
INFO:datacache.download:Downloading https://ftp.ensembl.org/pub/release-111/gtf/mus_musculus/Mus_musculus.GRCm39.111.gtf.gz to /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCm39/ensembl111/Mus_musculus.GRCm39.111.gtf.gz
INFO:pyensembl.download_cache:Fetching /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCm39/ensembl111/Mus_musculus.GRCm39.cdna.all.fa.gz from URL https://ftp.ensembl.org/pub/release-111/fasta/mus_musculus/cdna/Mus_musculus.GRCm39.cdna.all.fa.gz
INFO:datacache.download:Downloading https://ftp.ensembl.org/pub/release-111/fasta/mus_musculus/cdna/Mus_musculus.GRCm39.cdna.all.fa.gz to /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCm39/ensembl111/Mus_musculus.GRCm39.cdna.all.fa.gz
INFO:pyensembl.download_cache:Fetching /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCm39/ensembl

In [9]:
# Load data
if n_batches == 1:
    adata = sc.read_h5ad(f"{st_data_silver_folder_path}/{dataset_name}/{dataset_name}.h5ad")
else:
    adata = sc.read_h5ad(f"{st_data_silver_folder_path}/{dataset_name}/{dataset_name}_batch1.h5ad")

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '../../datasets/st_data/silver/merfish_mouse_brain/merfish_mouse_brain_batch1.h5ad', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [10]:
### Create token dict for nichejepa ###
with open(token_dictionary_file, "rb") as f:
    token_dict = pickle.load(f)

# Add cls token
if add_cls_tokens:
    def insert_at_index(d, index, key, value):
        items = list(d.items())
        items.insert(index, (key, value))
        new_d = dict(items)
        return new_d
    token_dict = insert_at_index(token_dict, 2, "<cls_cell>", 2)
    if separate_cell_and_neighborhood_tokens:
        token_dict = insert_at_index(token_dict, 3, "<cls_neighborhood>", 3)

# Remove genes not in dataset
gene_list = adata.var["ensembl_id"].tolist()
token_dict_copy = token_dict.copy()
for token in token_dict_copy.keys():
    if "ENS" in token and token not in gene_list:
        del(token_dict[token])
del(token_dict_copy)

# Separate gene tokens into cell and neighborhood tokens
if separate_cell_and_neighborhood_tokens:
    new_token_dict = {
        (key + "_cell" if "ENS" in key else key): value
        for value, key in enumerate(token_dict.keys())}
    new_token_dict_copy = new_token_dict.copy()
    i = 0
    for key in new_token_dict_copy.keys():
        if "ENS" in key:
            new_token_dict[key.replace("cell", "neighborhood")] = len(new_token_dict_copy) + i
            i += 1
    del(new_token_dict_copy)
else:
    new_token_dict = {
        key: value
        for value, key in enumerate(token_dict.keys())}

# Write to disk
with open(nichejepa_token_dictionary_file, "wb") as file:
    pickle.dump(new_token_dict, file)

NameError: name 'adata' is not defined

In [ ]:
# Load vocab
with open(nichejepa_token_dictionary_file, "rb") as file:
    token_dict = pickle.load(file)

token_dict_reverse = {value: key for key, value in token_dict.items()}

In [11]:
# NicheJEPA token dict
len(token_dict)

23899

In [12]:
n_gene_tokens = len([token for token in token_dict.keys() if "ENS" in token])

In [13]:
# First n tokens
n = 5
for key, value in list(token_dict.items())[:n]:
    print(f"{key}: {value}")

<pad>: 0
<mask>: 1
ENSMUSG00000000001: 2
ENSMUSG00000000003: 3
ENSMUSG00000000028: 4


In [14]:
if separate_cell_and_neighborhood_tokens and species == "homo_sapiens":
    # First cell gene token
    print(token_dict["ENSG00000000938_cell"])
    # First neighborhood gene token
    print(token_dict["ENSG00000000938_neighborhood"])

### 2.2 Compute Normalization Factors for Gene Counts

Corpus means, non-zero medians, or log means are used depending on the 'norm_method' specified in the CellNeighborhoodRankTokenizer (check source code).

In [ ]:
# Compute Corpus Means and Non-zero Medians for Cell and Neighborhood Genes
for norm_factor in norm_factors:
    # Load data
    if n_batches == 1:
        adata = sc.read_h5ad(f"{st_data_silver_folder_path}/{dataset_name}/{dataset_name}.h5ad")
        if type(adata.X) == sp._csr.csr_matrix:
            adata.X = adata.X.toarray()
    
        # Filter cells that did not pass QC
        if "filter_pass" in adata.obs.columns:
            filter_pass_idx = np.where([filter_pass == 1 for filter_pass in adata.obs["filter_pass"]])[0]
        else:
            print(f"adata has no column 'filter_pass'; utilizing all cells.")
            filter_pass_idx = np.array([i for i in range(adata.shape[0])])
        adata = adata[filter_pass_idx].copy()
        
        # Compute spatial neighborhood graph based on Visium spot diameter of 55 microns
        sq.gr.spatial_neighbors(adata,
                                coord_type="generic",
                                spatial_key="spatial",
                                radius=27.5,
                                )
        
        # Add self loops to spatial neighborhood graph
        adata.obsp["spatial_connectivities"].setdiag(1)
            
        # Compute sum of counts across each cell's neighborhood to get neighborhood counts per cell
        adata.layers["X_neighborhood"] = adata.obsp["spatial_connectivities"].T @ adata.X

        if norm_factor == "read_depth":
            # Normalize cell gene counts by read depth
            adata.X = adata.X / adata.X.sum(1).reshape(-1, 1) * 10000
        
            # Normalize neighborhood gene counts by read depth
            adata.layers["X_neighborhood"] = (
                adata.layers["X_neighborhood"] /
                adata.layers["X_neighborhood"].sum(1).reshape(-1, 1) * 10000)
        elif norm_factor == "cell_area":
            # Normalize cell gene counts by cell area
            adata.X = adata.X / adata.obs["cell_area"].values.reshape(-1, 1)
        
            # Compute neighborhood area
            adata.obs["neighborhood_cell_area"] = np.array(adata.obsp["spatial_connectivities"].T @ adata.obs["cell_area"].values.reshape(-1, 1))  
        
            # Normalize neighborhood counts
            adata.layers["X_neighborhood"] = (
                adata.layers["X_neighborhood"] /
                adata.obs["neighborhood_cell_area"].values.reshape(-1, 1))
        elif norm_factor == "prenormalized":
            pass
        elif norm_factor == "unnormalized":
            pass
    else:
        adata_batch_list = []
        for batch_idx in range(1, n_batches+1):
            adata_batch = sc.read_h5ad(f"{st_data_silver_folder_path}/{dataset_name}/{dataset_name}_batch{batch_idx}.h5ad")

            # Filter cells that did not pass QC
            if "filter_pass" in adata_batch.obs.columns:
                filter_pass_idx = np.where([filter_pass == 1 for filter_pass in adata_batch.obs["filter_pass"]])[0]
            else:
                print(f"adata has no column 'filter_pass'; utilizing all cells.")
                filter_pass_idx = np.array([i for i in range(adata_batch.shape[0])])
            adata_batch = adata_batch[filter_pass_idx].copy()

            # Compute spatial neighborhood graph based on Visium spot diameter of 55 microns
            sq.gr.spatial_neighbors(adata_batch,
                                    coord_type="generic",
                                    spatial_key="spatial",
                                    radius=27.5,
                                    )
    
            # Add self loops to spatial neighborhood graph
            adata_batch.obsp["spatial_connectivities"].setdiag(1)
                
            # Compute sum of counts across each cell's neighborhood to get neighborhood counts per cell
            adata_batch.layers["X_neighborhood"] = adata_batch.obsp["spatial_connectivities"].T @ adata_batch.X

            if norm_factor == "read_depth":
                # Normalize cell gene counts by read depth
                adata_batch.X = adata_batch.X / adata_batch.X.sum(1).reshape(-1, 1) * 10000
            
                # Normalize neighborhood gene counts by read depth
                adata_batch.layers["X_neighborhood"] = (
                    adata_batch.layers["X_neighborhood"] /
                    adata_batch.layers["X_neighborhood"].sum(1).reshape(-1, 1) * 10000)
            elif norm_factor == "cell_area":
                # Normalize cell gene counts by cell area
                adata_batch.X = adata_batch.X / adata_batch.obs["cell_area"].values.reshape(-1, 1)
            
                # Compute neighborhood area
                adata_batch.obs["neighborhood_cell_area"] = np.array(adata_batch.obsp["spatial_connectivities"].T @ adata_batch.obs["cell_area"].values.reshape(-1, 1))  
            
                # Normalize neighborhood counts
                adata_batch.layers["X_neighborhood"] = (
                    adata_batch.layers["X_neighborhood"] /
                    adata_batch.obs["neighborhood_cell_area"].values.reshape(-1, 1))
            elif norm_factor == "prenormalized":
                pass
            elif norm_factor == "unnormalized":
                pass
                    
            adata_batch_list.append(adata_batch)
        adata = ad.concat(adata_batch_list, join="inner", axis=0)

        # Combine spatial neighborhood graphs as disconnected components
        batch_connectivities = []
        len_before_batch = 0
        for i in range(len(adata_batch_list)):
            if i == 0: # first batch
                after_batch_connectivities_extension = sp.csr_matrix(
                    (adata_batch_list[0].shape[0],
                    (adata.shape[0] -
                    adata_batch_list[0].shape[0])))
                batch_connectivities.append(sp.hstack(
                    (adata_batch_list[0].obsp["spatial_connectivities"],
                    after_batch_connectivities_extension)))
            elif i == (len(adata_batch_list) - 1): # last batch
                before_batch_connectivities_extension = sp.csr_matrix(
                    (adata_batch_list[i].shape[0],
                    (adata.shape[0] -
                    adata_batch_list[i].shape[0])))
                batch_connectivities.append(sp.hstack(
                    (before_batch_connectivities_extension,
                    adata_batch_list[i].obsp["spatial_connectivities"])))
            else: # middle batches
                before_batch_connectivities_extension = sp.csr_matrix(
                    (adata_batch_list[i].shape[0], len_before_batch))
                after_batch_connectivities_extension = sp.csr_matrix(
                    (adata_batch_list[i].shape[0],
                    (adata.shape[0] -
                    adata_batch_list[i].shape[0] -
                    len_before_batch)))
                batch_connectivities.append(sp.hstack(
                    (before_batch_connectivities_extension,
                    adata_batch_list[i].obsp["spatial_connectivities"],
                    after_batch_connectivities_extension)))
            len_before_batch += adata_batch_list[i].shape[0]
        adata.obsp["spatial_connectivities"] = sp.vstack(batch_connectivities)

        # Copy ensembl IDs
        adata.var = adata_batch_list[0].var
        
    # Compute means and non-zero medians
    cell_gene_means = np.mean(adata.X, axis=0)
    masked_data = np.ma.masked_equal(adata.X, 0)
    cell_gene_nzmedians = np.ma.mean(masked_data, axis=0)
    cell_gene_nzmedians = cell_gene_nzmedians.filled(0)
    neighborhood_gene_means = np.mean(adata.layers["X_neighborhood"], axis=0)
    masked_data_neighborhood = np.ma.masked_equal(adata.layers["X_neighborhood"], 0)
    neighborhood_gene_nzmedians = np.ma.mean(masked_data_neighborhood, axis=0)
    neighborhood_gene_nzmedians = neighborhood_gene_nzmedians.filled(0)

    sc.pp.log1p(adata)
    sc.pp.log1p(adata, layer="X_neighborhood")
    cell_gene_logmeans = np.mean(adata.X, axis=0)
    neighborhood_gene_logmeans = np.mean(adata.layers["X_neighborhood"], axis=0)

    cell_gene_logmeans = np.array(cell_gene_logmeans).reshape(-1)
    cell_gene_nzmedians = np.array(cell_gene_nzmedians).reshape(-1)
    cell_gene_means = np.array(cell_gene_means).reshape(-1)
    neighborhood_gene_means = np.array(neighborhood_gene_means).reshape(-1)
    neighborhood_gene_nzmedians = np.array(neighborhood_gene_nzmedians).reshape(-1)
    neighborhood_gene_logmeans = np.array(neighborhood_gene_logmeans).reshape(-1)
    
    # Create norm factor dictionaries for tokenizer and store on disk
    cell_gene_means_dict = {}
    cell_gene_nzmedians_dict = {}
    cell_gene_logmeans_dict = {}
    neighborhood_gene_means_dict = {}
    neighborhood_gene_nzmedians_dict = {}
    neighborhood_gene_logmeans_dict = {}
    
    for (gene,
         cell_gene_mean,
         cell_gene_nzmedian,
         cell_gene_logmean,
         neighborhood_gene_mean,
         neighborhood_gene_nzmedian,
         neighborhood_gene_logmean) in zip(
            adata.var["ensembl_id"].values,
            cell_gene_means,
            cell_gene_nzmedians,
            cell_gene_logmeans,
            neighborhood_gene_means,
            neighborhood_gene_nzmedians,
            neighborhood_gene_logmeans):
        cell_gene_means_dict[gene] = cell_gene_mean
        cell_gene_nzmedians_dict[gene] = cell_gene_nzmedian
        cell_gene_logmeans_dict[gene] = cell_gene_logmean
        neighborhood_gene_means_dict[gene] = neighborhood_gene_mean
        neighborhood_gene_nzmedians_dict[gene] = neighborhood_gene_nzmedian
        neighborhood_gene_logmeans_dict[gene] = neighborhood_gene_logmean
    
    with open(cell_gene_means_dictionary_file.replace("gene", f"gene_{norm_factor}"), "wb") as file:
        pickle.dump(cell_gene_means_dict, file)
    with open(cell_gene_nzmedians_dictionary_file.replace("gene", f"gene_{norm_factor}"), "wb") as file:
        pickle.dump(cell_gene_nzmedians_dict, file)
    with open(cell_gene_logmeans_dictionary_file.replace("gene", f"gene_{norm_factor}"), "wb") as file:
        pickle.dump(cell_gene_logmeans_dict, file)
    with open(neighborhood_gene_means_dictionary_file.replace("gene", f"gene_{norm_factor}"), "wb") as file:
        pickle.dump(neighborhood_gene_means_dict, file)
    with open(neighborhood_gene_nzmedians_dictionary_file.replace("gene", f"gene_{norm_factor}"), "wb") as file:
        pickle.dump(neighborhood_gene_nzmedians_dict, file)
    with open(neighborhood_gene_logmeans_dictionary_file.replace("gene", f"gene_{norm_factor}"), "wb") as file:
        pickle.dump(neighborhood_gene_logmeans_dict, file)

In [ ]:
values = []
for key, value in cell_gene_means_dict.items():
    values.append(value)

np.mean(values)

In [ ]:
values = []
for key, value in neighborhood_gene_means_dict.items():
    values.append(value)

np.mean(values)

### 2.3 Run Tokenizer

In [ ]:
# Tokenize data with different normalization methods
for model_input_size in [n_gene_tokens * 2, # all genes
                         #512,
                        ]:
    for norm_method in ["analytic_pearson_residuals",
                        "mean",
                        "nzmedian",
                        "seurat_v3",
                        "shifted_logmean",
                        "shifted_log",
    ]:
        if norm_method == "analytic_pearson_residuals":
            output_file_prefix = f"{dataset_name}_{norm_method}_{model_input_size}"
    
            tk = tokenizers.CellNeighborhoodRankTokenizer(
                    model_input_size=model_input_size,
                    norm_method=norm_method,
                    separate_cell_and_neighborhood_tokens=separate_cell_and_neighborhood_tokens,
                    cell_gene_means_file=cell_gene_means_dictionary_file.replace("gene", f"gene_read_depth"),
                    cell_gene_nzmedians_file=cell_gene_nzmedians_dictionary_file.replace("gene", f"gene_read_depth"),
                    cell_gene_logmeans_file=cell_gene_logmeans_dictionary_file.replace("gene", f"gene_read_depth"),
                    neighborhood_gene_means_file=neighborhood_gene_means_dictionary_file.replace("gene", f"gene_read_depth"),
                    neighborhood_gene_nzmedians_file=neighborhood_gene_nzmedians_dictionary_file.replace("gene", f"gene_read_depth"),
                    neighborhood_gene_logmeans_file=neighborhood_gene_logmeans_dictionary_file.replace("gene", f"gene_read_depth"),
                    token_dictionary_file=nichejepa_token_dictionary_file,
                    custom_attr_name_dict=custom_attr_name_dict)
    
            tk.tokenize_data(input_directory=input_directory,
                             output_directory=output_directory,
                             output_file_prefix=output_file_prefix)
            
        elif norm_method != "analytic_pearson_residuals":
            for norm_factor in norm_factors:
                output_file_prefix = f"{dataset_name}_{norm_factor}_{norm_method}_{model_input_size}"
    
                tk = tokenizers.CellNeighborhoodRankTokenizer(
                        model_input_size=model_input_size,
                        norm_method=norm_method,
                        norm_factor=norm_factor,
                        separate_cell_and_neighborhood_tokens=separate_cell_and_neighborhood_tokens,
                        cell_gene_means_file=cell_gene_means_dictionary_file.replace("gene", f"gene_{norm_factor}"),
                        cell_gene_nzmedians_file=cell_gene_nzmedians_dictionary_file.replace("gene", f"gene_{norm_factor}"),
                        cell_gene_logmeans_file=cell_gene_logmeans_dictionary_file.replace("gene", f"gene_{norm_factor}"),
                        neighborhood_gene_means_file=neighborhood_gene_means_dictionary_file.replace("gene", f"gene_{norm_factor}"),
                        neighborhood_gene_nzmedians_file=neighborhood_gene_nzmedians_dictionary_file.replace("gene", f"gene_{norm_factor}"),
                        neighborhood_gene_logmeans_file=neighborhood_gene_logmeans_dictionary_file.replace("gene", f"gene_{norm_factor}"),
                        token_dictionary_file=nichejepa_token_dictionary_file,
                        custom_attr_name_dict=custom_attr_name_dict)
                    
                tk.tokenize_data(input_directory=input_directory,
                                 output_directory=output_directory,
                                 output_file_prefix=output_file_prefix)

### 2.4 Test Tokenizer

In [ ]:
norm_factor = norm_factors[0]
norm_method = "shifted_log"
model_input_size = n_gene_tokens * 2
tokenized_data_folder_path = f"{st_data_gold_folder_path}/{tokenizer}/{dataset_name}/{dataset_name}_{norm_factor}_{norm_method}_{model_input_size}.dataset"
dataset = load_from_disk(tokenized_data_folder_path, keep_in_memory=True)

In [ ]:
dataset

In [ ]:
print(dataset["gene_tokens_cell"][0][:10])
print(dataset["gene_tokens_neighborhood"][0][:10])
print(dataset["cell_types"][0])
print(dataset["niche_types"][0])
print(dataset["input_ids"][0][:10])
print(dataset["input_ids"][0][int(model_input_size/2):int(model_input_size/2)+10])

### 2.5 Evaluate Tokenizer

#### 2.5.1 Cell Classification

In [ ]:
# Load vocab
with open(nichejepa_token_dictionary_file, "rb") as file:
    token_dict = pickle.load(file)

token_dict_reverse = {value: key for key, value in token_dict.items()}

n_gene_tokens = len([token for token in token_dict.keys() if "ENS" in token])

In [ ]:
# Get scgpt gene embeddings for NicheJEPA input token ids
def get_classification_tokens(example, top_n, randomize=False):
    if randomize:
        random.shuffle(example["gene_tokens_cell"])
        random.shuffle(example["gene_tokens_neighborhood"])
    example[f"top_{top_n}_cell_tokens"] = [
        token_dict_reverse[token_id].replace("_cell", "")
        for token_id in example["gene_tokens_cell"][:top_n]]
    example[f"top_{top_n}_neighborhood_tokens"] = [
        token_dict_reverse[token_id].replace("_neighborhood", "")
        for token_id in example["gene_tokens_neighborhood"][:top_n]]                  
    return example

In [ ]:
# Test classification token retrieval
top_n = 10
model_input_size = n_gene_tokens * 2 # 512

norm_method = "shifted_log"
norm_factor = "read_depth" # "read_depth", "cell_area"
normalization = f"{norm_factor}_{norm_method}"
    
output_file_prefix = f"{dataset_name}_{normalization}_{model_input_size}"
tokenized_data_folder_path = f"{st_data_gold_folder_path}/{tokenizer}/{dataset_name}/{output_file_prefix}.dataset"
dataset = load_from_disk(tokenized_data_folder_path, keep_in_memory=True)
dataset = dataset.map(get_classification_tokens,
                      fn_kwargs={"top_n": top_n},
                      #cache_file_name=dataset_cache_file,
                     )

print(dataset[f"top_{top_n}_neighborhood_tokens"][0])
print("")
print(dataset[f"top_{top_n}_cell_tokens"][0])
print("")

top_n = n_gene_tokens
dataset = dataset.map(get_classification_tokens,
                      fn_kwargs={"top_n": top_n},
                      #cache_file_name=dataset_cache_file,
                     )

print(dataset[f"top_{top_n}_neighborhood_tokens"][0])
print("")
print(dataset[f"top_{top_n}_cell_tokens"][0])
print("")

##### 2.5.1.1 Baseline (Random Gene Embeddings)

Task: cell and niche type classification using randomly embedded top ranked genes.

In [ ]:
emsemblid_embedding_mapping = {}
for key in token_dict.keys():
   emsemblid_embedding_mapping[key] = np.random.randn(512)

In [ ]:
mode = "nopos"
top_n = 10 # n_gene_tokens, 30, 10

randomize = True # False, True
if randomize:
    random.seed(42)
    model_input_size = n_gene_tokens * 2
else:
    if top_n == n_gene_tokens:
        model_input_size = n_gene_tokens * 2
    else:
        model_input_size = n_gene_tokens * 2 # 512

scores = []

os.makedirs(f"{results_folder_path}/{tokenizer}/{dataset_name}", exist_ok=True)

for norm_method in ["analytic_pearson_residuals",
                    "mean",
                    "nzmedian",
                    "seurat_v3",
                    "shifted_logmean",
                    "shifted_log",
                   ]:
    for norm_factor in norm_factors:
        if norm_method == "analytic_pearson_residuals":
            normalization = f"{norm_method}"
            if norm_factor != "read_depth":
                continue
        if norm_method != "analytic_pearson_residuals":
            normalization = f"{norm_factor}_{norm_method}"
        
        output_file_prefix = f"{dataset_name}_{normalization}_{model_input_size}"
        tokenized_data_folder_path = f"{st_data_gold_folder_path}/{tokenizer}/{dataset_name}/{output_file_prefix}.dataset"
        dataset = load_from_disk(tokenized_data_folder_path, keep_in_memory=True)
        
        dataset = dataset.map(get_classification_tokens,
                              fn_kwargs={"top_n": top_n, "randomize": randomize},
                              #cache_file_name=dataset_cache_file,
                             )
    
        X_cell = np.array(
            [[emsemblid_embedding_mapping.get(token, np.random.randn(512)) for token in sequence] for
             sequence in dataset[f"top_{top_n}_cell_tokens"]])
        X_cell = np.mean(X_cell, axis=1) # aggregate across tokens
        X_cell = X_cell.reshape(X_cell.shape[0], -1)
    
        X_neighborhood = np.array(
            [[emsemblid_embedding_mapping.get(token, np.random.randn(512)) for token in sequence] for
             sequence in dataset[f"top_{top_n}_neighborhood_tokens"]])
        X_neighborhood = np.mean(X_neighborhood, axis=1) # aggregate across tokens
        X_neighborhood = X_neighborhood.reshape(X_neighborhood.shape[0], -1)
    
        for token_type in ["cell",
                           "neighborhood"
                          ]:
            for label_cat in label_cats:
                label_encoder = LabelEncoder()
                labels = np.array(
                    label_encoder.fit_transform(dataset[label_cat]))
    
                for random_state in range(10):
    
                    np.random.seed(random_state)
                
                    indices = np.arange(X_cell.shape[0])
                    np.random.shuffle(indices)
                    num_train = int(0.8 * len(indices))
                    train_indices = indices[:num_train]
                    test_indices = indices[num_train:]
                    labels_train = labels[train_indices]
                    labels_test = labels[test_indices]
                    if token_type == "cell":
                        X = X_cell
                    else:
                        X = X_neighborhood
                    X_train = X[train_indices]
                    X_test = X[test_indices]
                    
                    lr_clf = LogisticRegression()
                    lr_clf.fit(X_train, labels_train)
                    train_predictions = lr_clf.predict(X_train)
                    train_f1 = f1_score(labels_train, train_predictions, average="weighted")
                    train_score = lr_clf.score(X_train, labels_train)
                    test_predictions = lr_clf.predict(X_test)
                    test_f1 = f1_score(labels_test, test_predictions, average="weighted")
                    test_score = lr_clf.score(X_test, labels_test)
                    scores.append({"tokenizer": tokenizer,
                                   "normalization": normalization,
                                   "token_type": token_type,
                                   "clf_task": label_cat,
                                   "random_state": random_state,
                                   "clf_category": "train",
                                   "f1_score": train_f1,
                                   "accuracy": train_score})
                    scores.append({"tokenizer": tokenizer,
                                   "normalization": normalization,
                                   "token_type": token_type,
                                   "clf_task": label_cat,
                                   "random_state": random_state,
                                   "clf_category": "test",
                                   "f1_score": test_f1,
                                   "accuracy": test_score})
        
                    scores_df = pd.DataFrame(scores)
                    scores_df.to_csv(f"""{results_folder_path}/{tokenizer}/{dataset_name}/clf_scores_baseline_top{top_n}_{mode}{'_randomized' if randomize else ''}.csv""", index=False)

##### 2.5.1.2 Baseline (Random Gene Embeddings + Random Positional Embeddings)

In [ ]:
pos_embedding_mapping = {}
for key in range(n_gene_tokens):
   pos_embedding_mapping[key] = np.random.randn(512)

In [ ]:
mode = "randpos"
top_n = 10 # n_gene_tokens, 30, 10

if top_n == n_gene_tokens:
    model_input_size = n_gene_tokens * 2
else:
    model_input_size = n_gene_tokens * 2 # 512

scores = []

os.makedirs(f"{results_folder_path}/{tokenizer}/{dataset_name}", exist_ok=True)

for norm_method in ["analytic_pearson_residuals",
                    "mean",
                    "nzmedian",
                    "seurat_v3",
                    "shifted_logmean",
                    "shifted_log",
                   ]:
    for norm_factor in norm_factors:
        if norm_method == "analytic_pearson_residuals":
            normalization = f"{norm_method}"
            if norm_factor != "read_depth":
                continue
        if norm_method != "analytic_pearson_residuals":
            normalization = f"{norm_factor}_{norm_method}"
        
        output_file_prefix = f"{dataset_name}_{normalization}_{model_input_size}"
        tokenized_data_folder_path = f"{st_data_gold_folder_path}/{tokenizer}/{dataset_name}/{output_file_prefix}.dataset"
        dataset = load_from_disk(tokenized_data_folder_path, keep_in_memory=True)
        
        dataset = dataset.map(get_classification_tokens,
                              fn_kwargs={"top_n": top_n},
                              #cache_file_name=dataset_cache_file,
                             )
    
        X_cell = np.array(
            [[emsemblid_embedding_mapping.get(token, np.random.randn(512)) +
              pos_embedding_mapping[i] for i, token in enumerate(sequence)] for
             sequence in dataset[f"top_{top_n}_cell_tokens"]])
        X_cell = np.mean(X_cell, axis=1) # aggregate across tokens
        X_cell = X_cell.reshape(X_cell.shape[0], -1)
    
        X_neighborhood = np.array(
            [[emsemblid_embedding_mapping.get(token, np.random.randn(512)) +
              pos_embedding_mapping[i] for i, token in enumerate(sequence)] for
             sequence in dataset[f"top_{top_n}_neighborhood_tokens"]])
        X_neighborhood = np.mean(X_neighborhood, axis=1) # aggregate across tokens
        X_neighborhood = X_neighborhood.reshape(X_neighborhood.shape[0], -1)
    
        for token_type in ["cell",
                           "neighborhood"
                          ]:
            for label_cat in label_cats:
                label_encoder = LabelEncoder()
                labels = np.array(
                    label_encoder.fit_transform(dataset[label_cat]))
    
                for random_state in range(10):
    
                    np.random.seed(random_state)
                
                    indices = np.arange(X_cell.shape[0])
                    np.random.shuffle(indices)
                    num_train = int(0.8 * len(indices))
                    train_indices = indices[:num_train]
                    test_indices = indices[num_train:]
                    labels_train = labels[train_indices]
                    labels_test = labels[test_indices]
                    if token_type == "cell":
                        X = X_cell
                    else:
                        X = X_neighborhood
                    X_train = X[train_indices]
                    X_test = X[test_indices]
                    
                    lr_clf = LogisticRegression()
                    lr_clf.fit(X_train, labels_train)
                    train_predictions = lr_clf.predict(X_train)
                    train_f1 = f1_score(labels_train, train_predictions, average="weighted")
                    train_score = lr_clf.score(X_train, labels_train)
                    test_predictions = lr_clf.predict(X_test)
                    test_f1 = f1_score(labels_test, test_predictions, average="weighted")
                    test_score = lr_clf.score(X_test, labels_test)
                    scores.append({"tokenizer": tokenizer,
                                   "normalization": normalization,
                                   "token_type": token_type,
                                   "clf_task": label_cat,
                                   "random_state": random_state,
                                   "clf_category": "train",
                                   "f1_score": train_f1,
                                   "accuracy": train_score})
                    scores.append({"tokenizer": tokenizer,
                                   "normalization": normalization,
                                   "token_type": token_type,
                                   "clf_task": label_cat,
                                   "random_state": random_state,
                                   "clf_category": "test",
                                   "f1_score": test_f1,
                                   "accuracy": test_score})
        
                    scores_df = pd.DataFrame(scores)
                    scores_df.to_csv(f"{results_folder_path}/{tokenizer}/{dataset_name}/clf_scores_baseline_top{top_n}_{mode}.csv", index=False)

##### 2.5.1.3 Baseline (Random Gene Embeddings + Sinusoidal Positional Embeddings)

In [ ]:
pos_embed = get_1d_sincos_pos_embed_from_grid(
    512,
    np.arange(n_gene_tokens))

pos_embedding_mapping = {}
for key in range(n_gene_tokens):
   pos_embedding_mapping[key] = pos_embed[key]

In [ ]:
mode = "sinpos"
top_n = 10 # n_gene_tokens, 30, 10

if top_n == n_gene_tokens:
    model_input_size = n_gene_tokens * 2
else:
    model_input_size = n_gene_tokens * 2 # 512

scores = []

os.makedirs(f"{results_folder_path}/{tokenizer}/{dataset_name}", exist_ok=True)

for norm_method in ["analytic_pearson_residuals",
                    "mean",
                    "nzmedian",
                    "seurat_v3",
                    "shifted_logmean",
                    "shifted_log",
                   ]:
    for norm_factor in norm_factors:
        if norm_method == "analytic_pearson_residuals":
            normalization = f"{norm_method}"
            if norm_factor != "read_depth":
                continue
        if norm_method != "analytic_pearson_residuals":
            normalization = f"{norm_factor}_{norm_method}"
        
        output_file_prefix = f"{dataset_name}_{normalization}_{model_input_size}"
        tokenized_data_folder_path = f"{st_data_gold_folder_path}/{tokenizer}/{dataset_name}/{output_file_prefix}.dataset"
        dataset = load_from_disk(tokenized_data_folder_path, keep_in_memory=True)
        
        dataset = dataset.map(get_classification_tokens,
                              fn_kwargs={"top_n": top_n},
                              #cache_file_name=dataset_cache_file,
                             )
    
        X_cell = np.array(
            [[emsemblid_embedding_mapping.get(token, np.random.randn(512)) +
              pos_embedding_mapping[i] for i, token in enumerate(sequence)] for
             sequence in dataset[f"top_{top_n}_cell_tokens"]])
        X_cell = np.mean(X_cell, axis=1) # aggregate across tokens
        X_cell = X_cell.reshape(X_cell.shape[0], -1)
    
        X_neighborhood = np.array(
            [[emsemblid_embedding_mapping.get(token, np.random.randn(512)) +
              pos_embedding_mapping[i] for i, token in enumerate(sequence)] for
             sequence in dataset[f"top_{top_n}_neighborhood_tokens"]])
        X_neighborhood = np.mean(X_neighborhood, axis=1) # aggregate across tokens
        X_neighborhood = X_neighborhood.reshape(X_neighborhood.shape[0], -1)
    
        for token_type in ["cell",
                           "neighborhood"
                          ]:
            for label_cat in label_cats:
                label_encoder = LabelEncoder()
                labels = np.array(
                    label_encoder.fit_transform(dataset[label_cat]))
    
                for random_state in range(10):
    
                    np.random.seed(random_state)
                
                    indices = np.arange(X_cell.shape[0])
                    np.random.shuffle(indices)
                    num_train = int(0.8 * len(indices))
                    train_indices = indices[:num_train]
                    test_indices = indices[num_train:]
                    labels_train = labels[train_indices]
                    labels_test = labels[test_indices]
                    if token_type == "cell":
                        X = X_cell
                    else:
                        X = X_neighborhood
                    X_train = X[train_indices]
                    X_test = X[test_indices]
                    
                    lr_clf = LogisticRegression()
                    lr_clf.fit(X_train, labels_train)
                    train_predictions = lr_clf.predict(X_train)
                    train_f1 = f1_score(labels_train, train_predictions, average="weighted")
                    train_score = lr_clf.score(X_train, labels_train)
                    test_predictions = lr_clf.predict(X_test)
                    test_f1 = f1_score(labels_test, test_predictions, average="weighted")
                    test_score = lr_clf.score(X_test, labels_test)
                    scores.append({"tokenizer": tokenizer,
                                   "normalization": normalization,
                                   "token_type": token_type,
                                   "clf_task": label_cat,
                                   "random_state": random_state,
                                   "clf_category": "train",
                                   "f1_score": train_f1,
                                   "accuracy": train_score})
                    scores.append({"tokenizer": tokenizer,
                                   "normalization": normalization,
                                   "token_type": token_type,
                                   "clf_task": label_cat,
                                   "random_state": random_state,
                                   "clf_category": "test",
                                   "f1_score": test_f1,
                                   "accuracy": test_score})
        
                    scores_df = pd.DataFrame(scores)
                    scores_df.to_csv(f"{results_folder_path}/{tokenizer}/{dataset_name}/clf_scores_baseline_top{top_n}_{mode}.csv", index=False)

##### 2.5.1.4 scGPT

Task: cell and niche type classification using scGPT embedded top ranked genes.

In [ ]:
if "sim" not in dataset_name:
    ### Get scGPT gene embeddings for each gene Ensembl id ###
    
    # Set config
    pad_token = "<pad>"
    special_tokens = [pad_token, "<cls>", "<eoc>"]
    pad_value = -2
    n_input_bins = 51
    
    # Get vocab
    scgpt_vocab = GeneVocab.from_file(scgpt_vocab_file)
    for s in special_tokens:
        if s not in scgpt_vocab:
            scgpt_vocab.append_token(s)
    
    # Retrieve model parameters from config files
    with open(scgpt_model_config_file, "r") as f:
        model_configs = json.load(f)
    print(f"Resume model from {scgpt_model_file}, the model args will override the "
          f"config {scgpt_model_config_file}.")
    embsize = model_configs["embsize"]
    nhead = model_configs["nheads"]
    d_hid = model_configs["d_hid"]
    nlayers = model_configs["nlayers"]
    n_layers_cls = model_configs["n_layers_cls"]
    
    # Initialize model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    ntokens = len(scgpt_vocab)  # size of vocabulary
    model = TransformerModel(ntokens,
                             embsize,
                             nhead,
                             d_hid,
                             nlayers,
                             vocab=scgpt_vocab,
                             pad_value=pad_value,
                             n_input_bins=n_input_bins,)
    
    try:
        model.load_state_dict(torch.load(scgpt_model_file))
        print(f"Loading all model params from {scgpt_model_file}")
    except:
        model_dict = model.state_dict()
        pretrained_dict = torch.load(scgpt_model_file)
        pretrained_dict = {k: v
                           for k, v in pretrained_dict.items()
                           if k in model_dict and v.shape == model_dict[k].shape}
        for k, v in pretrained_dict.items():
            print(f"Loading params {k} with shape {v.shape}")
            model_dict.update(pretrained_dict)
            model.load_state_dict(model_dict)
    model.to(device)
    
    # Create mapping from Ensembl id to scGPT token id
    gene2idx = scgpt_vocab.get_stoi()
    
    metadata_df = pd.read_csv(scgpt_metadata_file)
    
    ensemblid_mapping = dict(zip(metadata_df["feature_name"], metadata_df["feature_id"]))
    ensemblid2idx = {ensemblid_mapping[k]: v for k, v in gene2idx.items() if k in ensemblid_mapping}
    
    # Retrieve the data-independent gene embeddings from scGPT
    gene_ids = np.array([id for id in ensemblid2idx.values()])
    gene_embeddings = model.encoder(torch.tensor(gene_ids, dtype=torch.long).to(device))
    gene_embeddings = gene_embeddings.detach().cpu().numpy()
    
    # Store gene embeddings with Ensembl id
    emsemblid_embedding_mapping = {}
    for i, (key, val) in enumerate(ensemblid2idx.items()):
        emsemblid_embedding_mapping[key] = gene_embeddings[i, :]

In [ ]:
if "sim" not in dataset_name:
    # Test scGPT embedding retrieval
    X_cell = np.array(
        [[emsemblid_embedding_mapping[token] for token in sequence] for
         sequence in dataset[f"top_{top_n}_cell_tokens"]])
    X_cell = X_cell.reshape(X_cell.shape[0], -1)
    
    X_neighborhood = np.array(
        [[emsemblid_embedding_mapping[token] for token in sequence] for
         sequence in dataset[f"top_{top_n}_neighborhood_tokens"]])
    X_neighborhood = X_neighborhood.reshape(X_neighborhood.shape[0], -1)
    
    print(X_cell)

In [ ]:
if "sim" not in dataset_name:
    model_input_size = n_gene_tokens * 2 # n_gene_tokens * 2, 512
    scores = []
    
    os.makedirs(f"{results_folder_path}/{tokenizer}/{dataset_name}", exist_ok=True)
    
    for norm_method in ["analytic_pearson_residuals",
                        "mean",
                        "nzmedian",
                        "seurat_v3",
                        "shifted_logmean",
                        "shifted_log",
                       ]:
        for norm_factor in norm_factors:
            if norm_method == "analytic_pearson_residuals":
                normalization = f"{norm_method}"
                if norm_factor == "cell_area":
                    continue
            if norm_method != "analytic_pearson_residuals":
                normalization = f"{norm_factor}_{norm_method}"
            
            output_file_prefix = f"{dataset_name}_{normalization}_{model_input_size}"
            tokenized_data_folder_path = f"{st_data_gold_folder_path}/{tokenizer}/{dataset_name}/{output_file_prefix}.dataset"
            dataset = load_from_disk(tokenized_data_folder_path, keep_in_memory=True)
    
            top_n = 10 # n_gene_tokens, 30, 10
            
            dataset = dataset.map(get_classification_tokens,
                                  fn_kwargs={"top_n": top_n},
                                  #cache_file_name=dataset_cache_file,
                                 )
        
            X_cell = np.array(
                [[emsemblid_embedding_mapping.get(token, np.random.randn(512)) for token in sequence] for
                 sequence in dataset[f"top_{top_n}_cell_tokens"]])
            X_cell = np.mean(X_cell, axis=1) # aggregate across tokens
            X_cell = X_cell.reshape(X_cell.shape[0], -1)
        
            X_neighborhood = np.array(
                [[emsemblid_embedding_mapping.get(token, np.random.randn(512)) for token in sequence] for
                 sequence in dataset[f"top_{top_n}_neighborhood_tokens"]])
            X_neighborhood = np.mean(X_neighborhood, axis=1) # aggregate across tokens
            X_neighborhood = X_neighborhood.reshape(X_neighborhood.shape[0], -1)
        
            for token_type in ["cell",
                               "neighborhood"
                              ]:
                for label_cat in label_cats:
                    label_encoder = LabelEncoder()
                    labels = np.array(
                        label_encoder.fit_transform(dataset[label_cat]))
        
                    for random_state in range(10):
        
                        np.random.seed(random_state)
                    
                        indices = np.arange(X_cell.shape[0])
                        np.random.shuffle(indices)
                        num_train = int(0.8 * len(indices))
                        train_indices = indices[:num_train]
                        test_indices = indices[num_train:]
                        labels_train = labels[train_indices]
                        labels_test = labels[test_indices]
                        if token_type == "cell":
                            X = X_cell
                        else:
                            X = X_neighborhood
                        X_train = X[train_indices]
                        X_test = X[test_indices]
                        
                        lr_clf = LogisticRegression()
                        lr_clf.fit(X_train, labels_train)
                        train_predictions = lr_clf.predict(X_train)
                        train_f1 = f1_score(labels_train, train_predictions, average="weighted")
                        train_score = lr_clf.score(X_train, labels_train)
                        test_predictions = lr_clf.predict(X_test)
                        test_f1 = f1_score(labels_test, test_predictions, average="weighted")
                        test_score = lr_clf.score(X_test, labels_test)
                        scores.append({"tokenizer": tokenizer,
                                       "normalization": normalization,
                                       "token_type": token_type,
                                       "clf_task": label_cat,
                                       "random_state": random_state,
                                       "clf_category": "train",
                                       "f1_score": train_f1,
                                       "accuracy": train_score})
                        scores.append({"tokenizer": tokenizer,
                                       "normalization": normalization,
                                       "token_type": token_type,
                                       "clf_task": label_cat,
                                       "random_state": random_state,
                                       "clf_category": "test",
                                       "f1_score": test_f1,
                                       "accuracy": test_score})
            
                        scores_df = pd.DataFrame(scores)
                        scores_df.to_csv(f"{results_folder_path}/{tokenizer}/{dataset_name}/clf_scores_scgpt_top{top_n}.csv", index=False)

### 2.6 Miscellaneous

In [ ]:
### Consolodidate all metrics into one file ###

# Define the folder containing CSV files
folder_path = "../datasets/results/cell_neighborhood_tokenizer/nanostring_cosmx_human_brain"

# List all CSV files in the folder
csv_files = [f for f in os.listdir(folder_path) if "clf_scores" in f and f.endswith('.csv')]

# Initialize an empty list to hold DataFrames
dataframes = []

# Load each CSV file into a DataFrame and append to the list
for file in csv_files:
    file_path = os.path.join(folder_path, file)
    df = pd.read_csv(file_path)
    dataframes.append(df)

# Concatenate all DataFrames vertically
combined_df = pd.concat(dataframes, ignore_index=True)

# Define the order of categories for each column
category_order_col1 = ['analytic_pearson_residuals',
                       'read_depth_mean',
                       'read_depth_nzmedian',
                       'read_depth_seurat_v3',
                       "read_depth_shifted_logmean",
                       "read_depth_shifted_log",
                       "cell_area_mean",
                       "cell_area_nzmedian",
                       "cell_area_seurat_v3",
                       "cell_area_shifted_logmean",
                       "cell_area_shifted_log"]  # Example order for column 1
category_order_col2 = ['cell', 'neighborhood']            # Example order for column 2
category_order_col3 = ['cell_types', 'niche_types']     # Example order for column 3

# Convert columns to categorical with defined order
combined_df['normalization'] = pd.Categorical(combined_df['normalization'], categories=category_order_col1, ordered=True)
combined_df['token_type'] = pd.Categorical(combined_df['token_type'], categories=category_order_col2, ordered=True)
combined_df['clf_task'] = pd.Categorical(combined_df['clf_task'], categories=category_order_col3, ordered=True)

# Sort the DataFrame by the three columns
sorted_df = combined_df.sort_values(by=['normalization', 'token_type', 'clf_task', "random_state"])
sorted_df.reset_index(inplace=True, drop=True)
unique_df = sorted_df.drop_duplicates(subset=['normalization', 'token_type', 'clf_task', 'random_state', 'clf_category'])
unique_df.reset_index(inplace=True, drop=True)

# Display the DataFrame
display(unique_df)

unique_df.to_csv(f"{results_folder_path}/{tokenizer}/{dataset_name}/clf_scores_scgpt.csv", index=False)

In [ ]:
len(unique_df)

In [ ]:
print(np.var([2, 4, 8]))
print(np.var([20, 40, 80]))
print(np.var([2, 4, 8] - np.array([np.mean([2, 20]), np.mean([4, 40]), np.mean([8, 80])])))
print(np.var([20, 40, 80] - np.array([np.mean([2, 20]), np.mean([4, 40]), np.mean([8, 80])])))

In [ ]:
print(np.var(np.log([2, 4, 8])))
print(np.var(np.log([20, 40, 80])))
print(np.var(np.log([2, 4, 8]) - np.array([np.mean(np.log([2, 20])), np.mean(np.log([4, 40])), np.mean(np.log([8, 80]))])))
print(np.var(np.log([20, 40, 80]) - np.array([np.mean(np.log([2, 20])), np.mean(np.log([4, 40])), np.mean(np.log([8, 80]))])))

In [ ]:
print(np.var(np.log([2, 4, 8])))
print(np.var(np.log([20, 40, 80])))
print(np.var(np.log([2, 4, 8]) / np.array([np.mean(np.log([2, 20])), np.mean(np.log([4, 40])), np.mean(np.log([8, 80]))])))
print(np.var(np.log([20, 40, 80]) / np.array([np.mean(np.log([2, 20])), np.mean(np.log([4, 40])), np.mean(np.log([8, 80]))])))